# Interactive Agent Session: Chapter 2 — LangChain HR Document Q&A (RAG)

This Jupyter notebook demonstrates a **Retrieval-Augmented Generation (RAG)** pipeline using LangChain. Instead of loading a real PDF, it uses a **self-contained simulated HR policy knowledge base** so the demo runs instantly without any file dependencies.

**How it works:**
1. HR policy text is chunked and turned into vector embeddings stored in an in-memory Chroma DB.
2. Each question retrieves only the **top-3 most relevant chunks** — keeping the LLM context lean.
3. The LLM answers strictly from the retrieved context, preventing hallucinations.

**Prerequisites:** Set your `OPENAI_API_KEY` below before running.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet langchain langchain_openai langchain_community langchain_chroma chromadb


In [ ]:
import os
from langchain.schema import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, HTML

# ── 1. API Key ────────────────────────────────────────────────────────────────
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# ── 2. Simulated HR Policy Knowledge Base (replaces a real PDF) ───────────────
#    In production: loader = PyPDFLoader("data/employee_handbook.pdf"); docs = loader.load_and_split()
HR_POLICY_TEXT = """
SECTION 1: PAID TIME OFF (PTO)
Full-time employees accrue 20 days of paid time off (PTO) per calendar year. PTO begins accruing from
the first day of employment at a rate of 1.67 days per month. Unused PTO may be carried over up to a
maximum of 10 days into the next calendar year. PTO must be requested at least 2 weeks in advance
through the HR portal. Approval is subject to team capacity.

SECTION 2: PARENTAL LEAVE
The company provides 16 weeks of fully paid maternity leave and 4 weeks of fully paid paternity leave
for eligible employees. Eligibility requires at least 12 months of continuous full-time employment prior
to the leave start date. Adoptive parents are entitled to the same benefits as biological parents.
Parental leave must be taken within the first year following birth or adoption.

SECTION 3: HEALTH INSURANCE
The company covers 90% of the employee health insurance premium for all full-time employees. Dental
and vision coverage is included in the base plan at no additional cost. Dependents may be added to the
plan; the company contributes 60% of dependent premiums. Open enrollment occurs every November
for coverage starting January 1.

SECTION 4: REMOTE WORK POLICY
Employees may work remotely up to 3 days per week with manager approval. Remote work agreements
must be documented through HR. Employees are required to be available and responsive during core
business hours (10 AM – 3 PM local time). A stipend of $600 per year is provided for home office setup.

SECTION 5: PERFORMANCE REVIEWS
Performance reviews are conducted twice per year — in June and December. Reviews include a
self-assessment, a manager assessment, and a peer feedback component. Compensation adjustments
and promotions are tied to the December review cycle. Employees receiving an 'Exceeds Expectations'
rating for two consecutive cycles are automatically considered for promotion.

SECTION 6: PROFESSIONAL DEVELOPMENT
Each employee receives an annual learning budget of $2,000 for courses, certifications, conferences,
and books. Requests must be submitted 30 days in advance and approved by the line manager and HR.
The company also offers 10 paid study days per year for employees pursuing industry certifications.

SECTION 7: GRIEVANCE & COMPLAINT PROCESS
Employees wishing to raise a formal complaint must submit a written statement to HR within 30 days
of the incident. HR will acknowledge receipt within 48 hours and initiate an investigation within 5
business days. Retaliation of any kind against an employee raising a grievance is a terminable offence.
"""

# ── 3. Split into chunks and vectorize ───────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
raw_docs = [Document(page_content=HR_POLICY_TEXT, metadata={"source": "employee_handbook_v3.pdf"})]
chunks = splitter.split_documents(raw_docs)

print(f"📚 Knowledge base split into {len(chunks)} retrievable chunks.")

try:
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    USE_REAL_LLM = True
    print("✅ Vector store built. LLM mode active.")
except Exception as e:
    USE_REAL_LLM = False
    print(f"⚠️  OpenAI key not set — running in demo simulation mode. ({e})")

# ── 4. Build the RAG Chain ───────────────────────────────────────────────────
SIMULATED_ANSWERS = {
    0: "You are entitled to **4 weeks (20 business days)** of fully paid paternity leave. Eligibility requires at least 12 months of continuous full-time employment.",
    1: "You may work remotely up to **3 days per week** with your manager's approval. You must remain available during core hours (10 AM – 3 PM local time) and a **$600/year home office stipend** is provided.",
    2: "Your annual learning budget is **$2,000** which covers courses, certifications, conferences, and books. You also get **10 paid study days** per year for certifications."
}

if USE_REAL_LLM:
    llm = ChatOpenAI(model="gpt-4-turbo")
    system_prompt = (
        "You are an authoritative HR Assistant. Answer employee questions strictly based on "
        "the retrieved HR policy context below. If the answer is not in the context, say "
        "'I could not find that information in the current policy handbook.' "
        "Keep answers concise, precise, and professional.\n\nContext:\n{context}"
    )
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    qa_chain = create_stuff_documents_chain(llm, prompt_template)
    rag_chain = create_retrieval_chain(retriever, qa_chain)

# ── 5. Gorgeous RAG Result UI ─────────────────────────────────────────────────
def render_rag_ui(qa_pairs: list):
    html = '''
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');
        .rag-wrap   { max-width: 820px; margin: 24px auto; font-family: 'Inter', sans-serif; background: #0f0f1a; padding: 32px; border-radius: 24px; box-shadow: 0 24px 60px rgba(0,0,0,0.6); border: 1px solid #1e1e3a; }
        .rag-title  { text-align:center; font-size: 22px; font-weight: 800; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 6px; background: linear-gradient(90deg, #f472b6, #818cf8); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
        .rag-sub    { text-align:center; color: #555; font-size: 13px; margin-bottom: 28px; letter-spacing: 0.5px; }
        .rag-card   { background: #131320; border: 1px solid #1e1e3a; border-radius: 18px; padding: 22px 26px; margin: 18px 0; transition: transform 0.15s, box-shadow 0.15s; }
        .rag-card:hover { transform: translateY(-2px); box-shadow: 0 12px 32px rgba(0,0,0,0.35); }
        .rag-q      { font-size: 13px; font-weight: 800; color: #818cf8; text-transform: uppercase; letter-spacing: 1.2px; margin-bottom: 10px; display: flex; align-items: center; gap: 8px; }
        .rag-q-text { font-size: 15px; color: #e2e8f0; font-weight: 600; border-left: 3px solid #6366f1; padding-left: 14px; margin: 0 0 16px 0; line-height: 1.6; }
        .rag-a-label{ font-size: 11px; font-weight: 800; color: #34d399; text-transform: uppercase; letter-spacing: 1.5px; margin-bottom: 8px; }
        .rag-a-text { background: linear-gradient(135deg, #064e3b, #065f46); border: 1px solid #059669; border-radius: 12px; padding: 14px 18px; color: #d1fae5; font-size: 14px; line-height: 1.7; }
        .rag-src    { margin-top: 12px; font-size: 11px; color: #4b5563; display: flex; align-items: center; gap: 6px; }
    </style>
    <div class="rag-wrap">
        <div class="rag-title">📋 HR Policy Q&amp;A Assistant</div>
        <div class="rag-sub">LangChain RAG Pipeline · Retrieval-Augmented Generation Demo</div>
    '''
    for i, pair in enumerate(qa_pairs):
        html += f'''
        <div class="rag-card">
            <div class="rag-q">🙋 Question {i+1}</div>
            <div class="rag-q-text">{pair["question"]}</div>
            <div class="rag-a-label">✅ HR Assistant Response</div>
            <div class="rag-a-text">{pair["answer"]}</div>
            <div class="rag-src">📄 Source: employee_handbook_v3.pdf · Retrieved {pair["chunks"]} policy chunks</div>
        </div>'''
    html += '</div>'
    display(HTML(html))

# ── 6. Run Three HR Questions ────────────────────────────────────────────────
questions = [
    "How many days of paid paternity leave am I entitled to?",
    "What is the remote work policy — how many days can I work from home?",
    "What is my annual budget for professional development and certifications?"
]

results = []
for i, question in enumerate(questions):
    try:
        if USE_REAL_LLM:
            resp = rag_chain.invoke({"input": question})
            answer = resp["answer"]
            num_chunks = len(resp.get("context", []))
        else:
            answer = SIMULATED_ANSWERS.get(i, "Simulated answer not available.")
            num_chunks = 3
        results.append({"question": question, "answer": answer, "chunks": num_chunks})
    except Exception as e:
        results.append({"question": question, "answer": f"Error: {e}", "chunks": 0})

render_rag_ui(results)
